# Dask-CUDA - 多 GPU 與大規模資料處理

> 當資料量超過單張 GPU 的 VRAM 時，Dask-CUDA 讓你用多張 GPU 或分散式運算來處理。

## GPU 加速系列 Notebooks

1. [Numba CUDA 基礎](./01-Introduction-to-cuda-python-with-numba.ipynb)
2. [GPU 加速概覽與環境建置](./02-GPU-Acceleration-Overview-and-Environment-Setup.ipynb)
3. [CuPy - GPU 版 NumPy](./03-CuPy-GPU-Accelerated-NumPy.ipynb)
4. [cuDF - GPU 版 pandas](./04-cuDF-GPU-Accelerated-DataFrames.ipynb)
5. [cuML - GPU 版 scikit-learn](./05-cuML-GPU-Accelerated-Machine-Learning.ipynb)
6. [cuGraph - GPU 版 NetworkX](./06-cuGraph-GPU-Accelerated-Graph-Analytics.ipynb)
7. **[Dask-CUDA 多 GPU 處理](./07-Dask-CUDA-Multi-GPU-and-Large-Scale-Processing.ipynb) ← 目前位置**
8. [RAPIDS 端到端實戰](./08-RAPIDS-End-to-End-Data-Analysis-Pipeline.ipynb)

---
## 0. 環境檢查

In [ ]:
import shutil
import subprocess

if shutil.which('nvidia-smi'):
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(result.stdout)
    GPU_AVAILABLE = True
else:
    print('未偵測到 NVIDIA GPU。')
    print('建議使用 Google Colab (免費 T4 GPU): https://colab.research.google.com')
    GPU_AVAILABLE = False

In [ ]:
# 安裝
# !pip install dask-cuda cudf-cu12 dask distributed

In [ ]:
import numpy as np
import pandas as pd
import time

# Dask (CPU)
import dask
import dask.dataframe as dd
print(f'Dask 版本: {dask.__version__}')

if GPU_AVAILABLE:
    import cudf
    import dask_cudf
    from dask_cuda import LocalCUDACluster
    from dask.distributed import Client
    print(f'cuDF 版本: {cudf.__version__}')

---
## 1. 什麼是 Dask？

**Dask** 是 Python 的分散式運算框架，讓你用熟悉的 pandas / NumPy API 處理超過記憶體大小的資料。

### 核心概念

```
pandas DataFrame (單一記憶體):        Dask DataFrame (分區):
┌──────────────────┐                ┌──────────────┐
│                  │                │  Partition 1  │ ← 各分區
│   全部資料       │                ├──────────────┤    可放在
│   必須放入       │    →           │  Partition 2  │    不同 GPU
│   單一記憶體     │                ├──────────────┤    或機器
│                  │                │  Partition 3  │
└──────────────────┘                └──────────────┘
```

### Dask + RAPIDS

| 組合 | 說明 |
|------|------|
| `dask.dataframe` | Dask + pandas (CPU 分散式) |
| `dask_cudf` | Dask + cuDF (GPU 分散式) |
| `dask_cuda.LocalCUDACluster` | 自動為每張 GPU 建立一個 worker |
| `cuml.dask` | Dask + cuML (分散式 GPU 機器學習) |

---
## 2. 啟動 Dask-CUDA Cluster

In [ ]:
if GPU_AVAILABLE:
    # 建立 LocalCUDACluster：每張 GPU 一個 worker
    cluster = LocalCUDACluster()
    client = Client(cluster)
    print(client)
    print(f'\nWorker 數量: {len(client.scheduler_info()["workers"])}')
    print(f'Dashboard: {client.dashboard_link}')
else:
    # CPU Dask cluster
    from dask.distributed import Client, LocalCluster
    cluster = LocalCluster(n_workers=2, threads_per_worker=2)
    client = Client(cluster)
    print('CPU Dask Cluster:')
    print(client)

---
## 3. Dask cuDF 基本操作

`dask_cudf` 的 API 與 `dask.dataframe` 幾乎相同，但底層使用 GPU。

In [ ]:
# 產生大量合成資料
N = 20_000_000  # 兩千萬筆
n_partitions = 10  # 分成 10 個分區

np.random.seed(42)

categories = ['電子', '服飾', '食品', '家居', '美妝']
regions = ['北部', '中部', '南部', '東部']

data = {
    'id': np.arange(N),
    'category': np.random.choice(categories, N),
    'region': np.random.choice(regions, N),
    'amount': np.random.randint(100, 100000, N).astype(np.int32),
    'quantity': np.random.randint(1, 20, N).astype(np.int32),
    'score': np.random.rand(N).astype(np.float32),
}

print(f'資料量: {N:,} 筆')
print(f'分區數: {n_partitions}')

In [ ]:
# 建立 Dask DataFrame
pdf = pd.DataFrame(data)

if GPU_AVAILABLE:
    # Dask cuDF DataFrame (GPU)
    ddf_gpu = dask_cudf.from_cudf(cudf.from_pandas(pdf), npartitions=n_partitions)
    print(f'Dask cuDF DataFrame:')
    print(f'  分區數: {ddf_gpu.npartitions}')
    print(f'  欄位:   {list(ddf_gpu.columns)}')
    print(f'  型別:   {type(ddf_gpu)}')
else:
    print('使用 CPU Dask DataFrame')

# Dask DataFrame (CPU)
ddf_cpu = dd.from_pandas(pdf, npartitions=n_partitions)
print(f'\nDask CPU DataFrame:')
print(f'  分區數: {ddf_cpu.npartitions}')

### 重要觀念：Lazy Evaluation

Dask 使用**延遲計算 (lazy evaluation)**：操作不會立即執行，而是建立計算圖，直到呼叫 `.compute()` 才真正執行。

```python
result = ddf.groupby('col').sum()  # 不會立即計算，只建立計算圖
result.compute()                    # 這時才真正執行
```

In [ ]:
# GroupBy 範例

# CPU Dask
start = time.time()
result_dask_cpu = ddf_cpu.groupby(['category', 'region']).agg({
    'amount': ['sum', 'mean'],
    'quantity': 'sum'
}).compute()  # .compute() 觸發實際運算
cpu_time = time.time() - start
print(f'Dask CPU groupby: {cpu_time:.4f} 秒')

# GPU Dask
if GPU_AVAILABLE:
    # 暖機
    _ = ddf_gpu.groupby('category').agg({'amount': 'sum'}).compute()

    start = time.time()
    result_dask_gpu = ddf_gpu.groupby(['category', 'region']).agg({
        'amount': ['sum', 'mean'],
        'quantity': 'sum'
    }).compute()
    gpu_time = time.time() - start
    print(f'Dask GPU groupby: {gpu_time:.4f} 秒')
    print(f'加速倍數: {cpu_time / gpu_time:.1f}x')

---
## 4. 處理超出 GPU 記憶體的資料

Dask-CUDA 的核心價值：**分區處理**讓你可以處理超過單張 GPU VRAM 的資料。

### 機制

```
50 GB 資料 → 分成 20 個分區 (每個 2.5 GB)
                ↓
GPU (16 GB VRAM) 依序處理每個分區
  ├─ 載入 Partition 1 → 處理 → 輸出
  ├─ 載入 Partition 2 → 處理 → 輸出
  └─ ... 直到全部完成
```

### 多 GPU 設定

```python
# 系統有多張 GPU 時，Dask-CUDA 自動分配
from dask_cuda import LocalCUDACluster
cluster = LocalCUDACluster()  # 自動偵測所有 GPU

# 例如 4 張 GPU：
# Worker 0 → GPU 0 → 處理 Partition 0, 4, 8, ...
# Worker 1 → GPU 1 → 處理 Partition 1, 5, 9, ...
# Worker 2 → GPU 2 → 處理 Partition 2, 6, 10, ...
# Worker 3 → GPU 3 → 處理 Partition 3, 7, 11, ...
```

---
## 5. 從檔案讀取分區資料

In [ ]:
import tempfile
import os

# 將資料寫成多個 Parquet 檔案（模擬大資料集）
tmp_dir = os.path.join(tempfile.gettempdir(), 'rapids_demo_parquet')
os.makedirs(tmp_dir, exist_ok=True)

# 分成多個 parquet 檔
chunk_size = N // n_partitions
for i in range(n_partitions):
    chunk = pdf.iloc[i * chunk_size : (i + 1) * chunk_size]
    chunk.to_parquet(os.path.join(tmp_dir, f'part_{i:03d}.parquet'), index=False)

print(f'已寫入 {n_partitions} 個 Parquet 檔案到 {tmp_dir}')
total_size = sum(
    os.path.getsize(os.path.join(tmp_dir, f))
    for f in os.listdir(tmp_dir) if f.endswith('.parquet')
)
print(f'總大小: {total_size / 1e6:.1f} MB')

In [ ]:
# 用 Dask 讀取分區 Parquet
parquet_path = os.path.join(tmp_dir, '*.parquet')

# CPU Dask
start = time.time()
ddf_file_cpu = dd.read_parquet(parquet_path)
result_file_cpu = ddf_file_cpu.groupby('category')['amount'].sum().compute()
cpu_time = time.time() - start
print(f'Dask CPU read + groupby: {cpu_time:.4f} 秒')

# GPU Dask
if GPU_AVAILABLE:
    start = time.time()
    ddf_file_gpu = dask_cudf.read_parquet(parquet_path)
    result_file_gpu = ddf_file_gpu.groupby('category')['amount'].sum().compute()
    gpu_time = time.time() - start
    print(f'Dask GPU read + groupby: {gpu_time:.4f} 秒')
    print(f'加速倍數: {cpu_time / gpu_time:.1f}x')

In [ ]:
# 清理暫存檔
import shutil as sh
sh.rmtree(tmp_dir, ignore_errors=True)
print('暫存檔已清理')

---
## 6. Dask-CUDA 記憶體管理

### RMM (RAPIDS Memory Manager)

```python
from dask_cuda import LocalCUDACluster

# 使用記憶體池提升效能
cluster = LocalCUDACluster(
    rmm_pool_size='10GB',    # 預分配記憶體池
    rmm_async=True,          # 非同步記憶體管理
)
```

### 監控記憶體

```python
# 查看每個 worker 的 GPU 記憶體使用
client.run(lambda: {
    'used': cp.cuda.Device().mem_info[1] - cp.cuda.Device().mem_info[0],
    'total': cp.cuda.Device().mem_info[1]
})
```

### Spilling (溢出到 CPU)

當 GPU 記憶體不足時，Dask-CUDA 可以自動將資料暫存到 CPU 記憶體：

```python
cluster = LocalCUDACluster(
    device_memory_limit='12GB',  # GPU 記憶體上限
    memory_limit='64GB',         # CPU 記憶體上限
)
```

---
## 7. 分散式 ML 訓練 (cuml.dask)

cuML 的 Dask 介面允許跨多 GPU 訓練模型。

```python
from cuml.dask.cluster import KMeans as daskKMeans
from cuml.dask.decomposition import PCA as daskPCA

# 分散式 K-Means
kmeans = daskKMeans(n_clusters=10, client=client)
kmeans.fit(dask_cudf_df)  # 自動分散到多 GPU
labels = kmeans.predict(dask_cudf_df)

# 分散式 PCA
pca = daskPCA(n_components=10, client=client)
transformed = pca.fit_transform(dask_cudf_df)
```

### 支援的分散式演算法

- KMeans
- PCA / TruncatedSVD
- Linear Regression / Ridge / Lasso
- Random Forest (Classifier / Regressor)
- Nearest Neighbors

---
## 8. 何時使用 Dask-CUDA

### 決策流程

```
資料放得進單 GPU VRAM？
  ├─ 是 → 用 cuDF / cuML（更簡單、更快）
  └─ 否 → 有多張 GPU？
            ├─ 是 → Dask-CUDA + dask_cudf (平行處理)
            └─ 否 → Dask-CUDA 單 GPU + spilling (分區處理)
                     或考慮 Dask CPU (成本更低)
```

### 比較

| 場景 | 建議 |
|------|------|
| 資料 < GPU VRAM | cuDF / cuML（直接使用，不需 Dask） |
| 資料 > GPU VRAM, 單 GPU | Dask-CUDA + spilling |
| 資料 > GPU VRAM, 多 GPU | Dask-CUDA + LocalCUDACluster |
| 多節點叢集 | Dask-CUDA + 分散式 scheduler |
| 預算有限 | Dask CPU (純 CPU 分散式) |

In [ ]:
# 關閉 cluster
client.close()
cluster.close()
print('Cluster 已關閉')

---
## 參考資源

- [Dask-CUDA 官方文件](https://docs.rapids.ai/api/dask-cuda/stable/)
- [dask_cudf 文件](https://docs.rapids.ai/api/cudf/stable/dask_cudf/)
- [Dask 官方文件](https://docs.dask.org/)
- [cuml.dask 分散式 ML](https://docs.rapids.ai/api/cuml/stable/api/#distributed)

> **下一步：** 前往 [08-RAPIDS-End-to-End-Data-Analysis-Pipeline.ipynb](./08-RAPIDS-End-to-End-Data-Analysis-Pipeline.ipynb) 體驗 RAPIDS 全流程實戰。